In [1]:
import os
import json
import requests
from dotenv import load_dotenv

from openai import OpenAI

load_dotenv()

API = os.getenv("GROQ_API_KEY")
MODEL = "llama-3.1-8b-instant"
URL = "https://api.groq.com/openai/v1"

client = OpenAI(api_key=API, base_url=URL)

In [2]:
with open("test_queries.json") as f:
    queries = json.load(f)

len(queries)

20

In [3]:
def llm_judge(expected, actual):
    prompt = f"""judge if the actual answer matches with the expected answer.

Expected: {expected}
Actual: {actual}

Reply with only: MATCH or NOT_MATCH
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return "MATCH" in response.choices[0].message.content

In [4]:
CACHE_FILE = "eval_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        cache = json.load(f)
else:
    cache = []

In [5]:
if not cache:
    for q in queries:
        payload = {
            "query": q["query"],
            "doc_ids": [q["doc_id"]] if q.get("doc_id") else []
        }

        res = requests.post("http://localhost:5000/api/v1/query", json=payload)
        res_data = res.json()

        cache.append({
            "query": q["query"],
            "expected": q["expected_answer"],
            "answer": res_data.get("answer", ""),
            "retrieved_chunks": res_data.get("retrieved_chunks", []),
            "relevant_chunk_id": q.get("relevant_chunk_id"),
            "relevant_chunk": q.get("relevant_chunk")
        })

    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

### Evaluation loop

In [6]:
results = []
retrieval_total = 0
retrieval_success = 0

for item in cache:

    expected = item["expected"]
    actual = item["answer"]

    # Accuracy
    if expected is None or expected == "NOT_FOUND":
        is_correct = "NOT_FOUND" in actual or "Cannot answer" in actual
    else:
        is_correct = llm_judge(expected, actual)

    # Recall@K
    recall_hit = False
    if item.get("relevant_chunk"):
        retrieval_total += 1
        if item.get("relevant_chunk_id") in item["retrieved_chunks"]:
            retrieval_success += 1
            recall_hit = True

    results.append({
        "question": item["query"],
        "expected": expected,
        "got": actual,
        "correct": is_correct,
        "recall_hit": recall_hit
    })

### Metrics

In [7]:
accuracy = sum(r["correct"] for r in results) / len(results)
recall_at_k = retrieval_success / retrieval_total if retrieval_total else 0

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Recall@K: {recall_at_k*100:.2f}%")

Accuracy: 75.00%
Recall@K: 86.67%


### Debug failures

In [8]:
for r in results:
    if not r["correct"]:
        print("\n--- FAIL ---")
        print("Q:", r["question"])
        print("Expected:", r["expected"])
        print("Got:", r["got"])


--- FAIL ---
Q: What is the exact population of Rachel, Nevada?
Expected: None
Got: 

--- FAIL ---
Q: Who was the first person to contract COVID-19?
Expected: None
Got: 

--- FAIL ---
Q: What was the name of the ship that sank during the 1999 Odisha cyclone?
Expected: None
Got: 

--- FAIL ---
Q: How many UFOs are stored at Area 51?
Expected: None
Got: 

--- FAIL ---
Q: What is the current temperature at Groom Lake?
Expected: None
Got: 


### Quick Insight

In [9]:
len([r for r in results if r["recall_hit"]]), "retrieval hits"

(13, 'retrieval hits')